# Assessment of instance segmentation performance

In [ ]:
%matplotlib inline

In [ ]:
import os
import glob
from pathlib import Path

import matplotlib.pyplot as plt
import colorcet as cc
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import seaborn as sb

from skimage import io
import skimage.morphology
import skimage.segmentation

import evaluation

## Set data paths

In [ ]:
object = 'fibre'

# Path containing the ground truth (label images) for the validation set
ground_truth_path = os.path.join(r'data_evaluation\ground_truth', object)

# Path containing the prediction (label images) for the validation set of several models to be compared
model0_path = os.path.join(r'data_evaluation\zhang_alpha02_semantic', object)
model1_path = os.path.join(r'data_evaluation\lee_alpha01_semantic', object)
model2_path = os.path.join(r'data_evaluation\lee_alpha02_semantic', object)
model3_path = os.path.join(r'data_evaluation\lee_alpha04_semantic', object)
model4_path = os.path.join(r'data_evaluation\lee_alpha06_semantic', object)


model_paths = [model0_path, model1_path, model2_path, model3_path, model4_path] #include paths directly
#model_paths = []
#model_paths.append(model1_path)

# Creates a list of the model names, defined as the last element on the model path
model_names = []

for path in model_paths:
    name = Path(path).parts[1]
    model_names.append(name)



print(model_names)

## Auxiliary visualization function

In [ ]:
# Get glasbey cmap
l=cc.cm.glasbey_bw_minc_20_minl_30_r.colors            
l[0]=[0,0,0]
cmap_glasbey = LinearSegmentedColormap.from_list('my_list', l, N=1000)

In [ ]:
# Display prediction along with segmentation to visualize errors

def show(ground_truth, prediction, model_name, threshold=0.5, image_name="N"):
    
    # Compute Intersection over Union
    IOU = evaluation.intersection_over_union(ground_truth, prediction)
    
    # Create diff map
    diff = np.zeros(ground_truth.shape + (3,))
    A = ground_truth.copy()
    B = prediction.copy()
    A[A > 0] = 1
    B[B > 0] = 1
    D = A - B
    
    # Object-level errors
    C = IOU.copy()
    C[C>=threshold] = 1
    C[C<threshold] = 0
    missed = np.where(np.sum(C,axis=1) == 0)[0]
    extra = np.where(np.sum(C,axis=0) == 0)[0]

    for m in missed:
        diff[ground_truth == m+1, 0] = 1
    for e in extra:
        diff[prediction == e+1, 2] = 1
    
    # Display figures
    fig, ax = plt.subplots(1, 4, figsize=(18,6))
    fig.suptitle(model_name, fontsize=16)
    ax[0].imshow(ground_truth, cmap=cmap_glasbey, interpolation='nearest')
    ax[0].set_title("True objects:"+str(len(np.unique(ground_truth))))
    ax[1].imshow(diff)
    ax[1].set_title("Missed:"+str(len(missed)))
    ax[2].imshow(prediction, cmap=cmap_glasbey, interpolation='nearest')
    ax[2].set_title("Predicted objects:"+str(len(np.unique(prediction))))
    ax[3].imshow(IOU)
    ax[3].set_title(image_name)

## Run the evaluation

In [ ]:
def evaluate(ground_truth_path, model_path):
    all_images =[]
    for file in glob.glob(ground_truth_path + "\*.tif"):
        all_images.append(os.path.basename(file))
    #print(all_images)
    
    seg_metrics_table = pd.DataFrame(columns=["Image_Name", "Threshold", "F1", "Precision", "Recall", "Jaccard", "TP", "FP", "FN"])
    fn_features_table = pd.DataFrame(columns=["False_Negative", "Area", "Solidity", "Circularity", "Aspect_Ratio"])
    split_merge_iou_table = pd.DataFrame(columns=["Image_Name", "Merges", "Splits"])
    split_merge_iopiot_table = pd.DataFrame(columns=["Image_Name", "Merges", "Splits"])
    targ_pred = pd.DataFrame(columns=["Image_Name", "Targets", "Predictions"])

    for image_name in all_images:
        # Load ground truth data
        img_filename = os.path.join(ground_truth_path, image_name)
        ground_truth = io.imread(img_filename)
            
        # Load predictions model
        pred_model_filename = os.path.join(model_path, image_name)
        prediction_model = io.imread(pred_model_filename)
        
        # Transform to label matrix
        # NOTE: skimage.morphology.label will split objects made of multiparticle masks
        #ground_truth = skimage.morphology.label(ground_truth)
        #prediction_model = skimage.morphology.label(prediction_model)
        
        # Compute evaluation metrics
        seg_metrics_table = evaluation.multi_threshold_results(
            ground_truth, 
            prediction_model, 
            seg_metrics_table, 
            image_name
        )
        
        fn_features_table = evaluation.false_negative_features(
            ground_truth, 
            prediction_model, 
            fn_features_table,
        )
        
        split_merge_iou_table = evaluation.split_merge_iou(
            ground_truth, 
            prediction_model, 
            split_merge_iou_table, 
            image_name
        )

        split_merge_iopiot_table = evaluation.split_merge_iopiot(
            ground_truth,
            prediction_model,
            split_merge_iopiot_table,
            image_name
        )

        targ_pred = evaluation.targets_and_predictions(
            ground_truth,
            prediction_model,
            targ_pred,
            image_name
        )
        
        # Display an example image
        if image_name == all_images[0]:
            model_name = Path(model_path).stem
            show(ground_truth, prediction_model, model_name=model_name, image_name=image_name)
        
    return seg_metrics_table, fn_features_table, split_merge_iou_table, split_merge_iopiot_table, targ_pred

In [ ]:
seg_metrics_table = []
fn_features_table = []
split_merge_iou_table = []
split_merge_iopiot_table = []
targ_pred = []

for path in model_paths:
    out_A, out_B, out_C, out_D, out_E = evaluate(ground_truth_path, path)
    seg_metrics_table.append(out_A)
    fn_features_table.append(out_B)
    split_merge_iou_table.append(out_C)
    split_merge_iopiot_table.append(out_D)
    targ_pred.append(out_E)

In [ ]:
# Count total true (target) and predicted objects

for i in range(len(model_paths)-1):
    assert(np.sum(targ_pred[i]["Targets"]) == np.sum(targ_pred[i+1]["Targets"]))

true_objects_total = np.sum(targ_pred[0]["Targets"])

predicted_objects_total = []
for df in targ_pred:
    predictions = np.sum(df["Predictions"])
    predicted_objects_total.append(predictions)

print(predicted_objects_total)

## Report of results

In [ ]:
# Display accuracy results
def avr_perf(results):
    results = results[results["Threshold"] < 0.95]

    average_performance = results.groupby("Threshold").mean().reset_index()

    R = results.groupby("Image_Name").mean().reset_index()
    g = sb.jointplot(data=R[R["F1"] > 0.4], x="Jaccard", y="F1", xlim=[0.8,1], ylim=[0.6,1])
    g.fig.set_figwidth(4)
    g.fig.set_figheight(8)

    #average_performance
    R.sort_values(by="F1",ascending=False).loc[0, "Image_Name"]

    return average_performance

In [ ]:
average_performance = []

for df in seg_metrics_table:
    performances = avr_perf(df)
    average_performance.append(performances)

average_performance_concat = pd.concat(average_performance, keys=model_names, names=["Model"])

In [ ]:
# Plot instance segmentation metrics
fig, axs=plt.subplots(1,3,figsize=(16,4))
fig.suptitle('Metrics to assess instance segmentation performance')
g1=sb.lineplot(
    data=average_performance_concat,
    x="Threshold",
    y="F1",
    hue="Model",
    err_style='band',
    markers=True,
    dashes=False,
    ax=axs[0]
    )
axs[0].set_ylim(0,1)
g2=sb.lineplot(
    data=average_performance_concat,
    x="Threshold",
    y="Precision",
    hue="Model",
    err_style='band',
    markers=True,
    dashes=False,
    ax=axs[1])
axs[1].set_ylim(0,1)
g3=sb.lineplot(
    data=average_performance_concat,
    x="Threshold",
    y="Recall",
    hue="Model",
    err_style='band',
    markers=True,
    dashes=False,
    ax=axs[2])
axs[2].set_ylim(0,1)
plt.show()

average_performance_concat.to_csv(object + '__IoUrange_scores.csv')

average_performance_concat

In [ ]:
# Calculate f1, precision and recall averages along the different IoU thresholds
# Since the Jaccard Index assesses semantic segmentation, it's constant along the IoU thresholds

average_F1_score = []
average_precision = []
average_recall = []
jaccard_index = []

for df in average_performance:
    f1_mean = df["F1"].mean()
    precision_mean = df["Precision"].mean()
    recall_mean = df["Recall"].mean()
    jaccard_mean = df["Jaccard"].mean()

    average_F1_score.append(f1_mean)
    average_precision.append(precision_mean)
    average_recall.append(recall_mean)
    jaccard_index.append(jaccard_mean)


# Initialises empty dictionary using model names as keys (column header)
dict_averages = dict.fromkeys(model_names)

# Fills the dictionary using the averages of each metric as values
keys = range(len(model_names))
for i in keys:
        dict_averages[model_names[i]] = [average_F1_score[i], average_precision[i], average_recall[i], jaccard_index[i]]

# Create a pandas DataFrame using the metric names as indexes (row header)
index = ['F1', 'Precision', 'Recall', 'Jaccard']
average_df = pd.DataFrame(data=dict_averages, index=index)

# Plot with matplotlib
fig_average = average_df.plot.bar()
plt.suptitle(object.capitalize() + ' average scores')
fig_average.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.xticks(rotation = 360)
plt.savefig(object + '_scores.pdf') 
plt.show()

average_df.to_csv(object + '_scores.csv')

# Display DataFrame
average_df

In [ ]:
# Summarize False Negatives by morphometric feature
def group_fn(fn_features_table, feature, bins, labels):
    false_negatives_only = fn_features_table[fn_features_table["False_Negative"] == 1]

    groups_fn = false_negatives_only.groupby(
        pd.cut(
            false_negatives_only[feature], 
            bins=bins, # feature values (as a list) to set intervals
            labels=labels # labels to tag the intervals (as a list)
        )
    )["False_Negative"].sum()

    false_nagatives_local = fn_features_table.copy()
    false_nagatives_local["False_Negative"] = 1
    groups_all = false_nagatives_local.groupby(
        pd.cut(
            false_nagatives_local[feature], 
            bins=bins, # feature values (as a list) to set intervals
            labels=labels # labels to tag the intervals (as a list)
        )
    )["False_Negative"].sum()

    return groups_fn, groups_all



# Plot False Negatives by morphometric feature to compare models
def fn_fraction_plot(fn_features_table, feature, bin_intervals, labels, model_names, true_objects_total):
    feature_intervals_fn = []
    feature_intervals_all = []

    for df in fn_features_table:
        inter_fn, iter_all = group_fn(fn_features_table=df, feature=feature, bins=bin_intervals, labels=labels)
        feature_intervals_fn.append(inter_fn.tolist())
        feature_intervals_all.append(iter_all.tolist())
    
    percentage_total = []
    percentage_interval = []
    matrix = np.array(feature_intervals_fn)

    for i in range (len(labels)):
        pt_total = np.array(matrix[:,i]) / true_objects_total * 100
        pt_interval = np.array(matrix[:,i]) / feature_intervals_all[0][i] * 100
        percentage_total.append(pt_total)
        percentage_interval.append(pt_interval)
    
    # Initialises empty dictionary using model names as keys (column header)
    data1 = dict.fromkeys(labels)
    data2 = dict.fromkeys(labels)

    # Fills the dictionary using the averages of each metric as values
    label_keys = range(len(labels))
    for i in label_keys:
            data1[labels[i]] = percentage_total[i]
            data2[labels[i]] = percentage_interval[i]
    
    index = model_names

    df_intervals_total = pd.DataFrame(data=data1, index=index)
    df_intervals_per_interval = pd.DataFrame(data=data2, index=index)

    # Creates a figure with 2 horizontal subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
    title = 'False negatives by ' + feature.lower()
    fig.suptitle(title)

    df_intervals_total.plot.bar(ax=ax1, stacked=True)
    df_intervals_per_interval.plot.bar(ax=ax2, stacked=False)
    
    ax1.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax1.legend().set_visible(False)
    ax2.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax2.legend().set_visible(False)
    ax1.set_ylabel('Percentage of missed objects')
    ax2.set_ylabel('Percentage of missed objects (intragroupal)')
    ax1.tick_params('x', labelrotation=360)
    ax2.tick_params('x', labelrotation=360)
    fig.tight_layout(pad=1, w_pad=4., h_pad=4.)
    plt.show()

    return df_intervals_total, df_intervals_per_interval

In [ ]:
# FN grouped by area
fn_fraction_plot(
    fn_features_table=fn_features_table,
    feature="Area",
    bin_intervals=[0,500,2000,5000,40000],
    labels=["Tiny","Small","Normal","Large"],
    model_names=model_names,
    true_objects_total=true_objects_total)

In [ ]:
# FN grouped by solidity
fn_fraction_plot(
    fn_features_table=fn_features_table,
    feature="Solidity",
    bin_intervals=[0.0,0.88,0.93,0.96,1.0],
    labels=["Low","Mid-low","Mid-high","High"],
    model_names=model_names,
    true_objects_total=true_objects_total)

In [ ]:
# FN grouped by circularity
fn_fraction_plot(
    fn_features_table=fn_features_table,
    feature="Circularity",
    bin_intervals=[0.0,0.8,0.9,0.95,1.0],
    labels=["Low","Mid-low","Mid-high","High"],
    model_names=model_names,
    true_objects_total=true_objects_total)

In [ ]:
# FN grouped by aspect ratio
fn_fraction_plot(
    fn_features_table=fn_features_table,
    feature="Aspect_Ratio",
    bin_intervals=[1.0,1.1,1.2,1.4,10.],
    labels=["Low","Mid-low","Mid-high","High"],
    model_names=model_names,
    true_objects_total=true_objects_total)

In [ ]:
# Summarize splits and merges

# NOTE: The fraction of splits and merges is calculated in a different way

# SPLITS: a split event is defined as a target object overlapping (IoU > 0.1) more
# than one predicted objects. Therefore, the fraction of splits is calculated as the
# split-target ratio

# MERGES: a merge event is defined as a predicted object overlapping (IoU > 0.1)
# more than one target object. Therefore, the fraction of merges is calculated as
# the merge-predicted ratio, which is different for each model!

splits = []
merges = []
splits_percent = []
merges_percent = []
i=0

for df in split_merge_iou_table:
    splits.append(np.sum(df["Splits"]))
    merges.append(np.sum(df["Merges"]))
    sp = np.sum(df["Splits"]) / true_objects_total * 100
    mg = np.sum(df["Merges"]) / predicted_objects_total[i] * 100
    splits_percent.append(sp)
    merges_percent.append(mg)
    i+=1

d={'splits': splits_percent, 'merges': merges_percent}
index = model_names

df_seg_errors = pd.DataFrame(data=d, index=index)
fig_seg_errors = df_seg_errors.plot.bar()
plt.suptitle('Split and merged objects (IoU > 0.1)')
fig_seg_errors.set_ylabel('Split-target and Merge-predicted percentages')
fig_seg_errors.yaxis.set_major_formatter(mtick.PercentFormatter())
plt.xticks(rotation = 360)
plt.show()

d2={'splits': splits, '%_splits': splits_percent, 'merges': merges, '%_merges': merges_percent}
df_seg_errors_total_and_percent = pd.DataFrame(data=d2, index=index)
df_seg_errors_total_and_percent

In [ ]:
splits = []
merges = []
splits_percent = []
merges_percent = []
i=0

for df in split_merge_iopiot_table:
    splits.append(np.sum(df["Splits"]))
    merges.append(np.sum(df["Merges"]))
    sp = np.sum(df["Splits"]) / true_objects_total * 100
    mg = np.sum(df["Merges"]) / predicted_objects_total[i] * 100
    splits_percent.append(sp)
    merges_percent.append(mg)
    i+=1

d={'splits': splits_percent, 'merges': merges_percent}
index = model_names

df_seg_errors = pd.DataFrame(data=d, index=index)
fig_seg_errors = df_seg_errors.plot.bar()
plt.suptitle('Split (IOP > 0.2) and merged (IOT > 0.2) objects')
fig_seg_errors.set_ylabel('Split-target and Merge-predicted percentages')
fig_seg_errors.yaxis.set_major_formatter(mtick.PercentFormatter())
plt.xticks(rotation = 360)
plt.show()

d2={'splits': splits, '%_splits': splits_percent, 'merges': merges, '%_merges': merges_percent}
df_seg_errors_total_and_percent = pd.DataFrame(data=d2, index=index)
df_seg_errors_total_and_percent

In [ ]:
# Report false positives

false_positives = []
i=0

for df in seg_metrics_table:
    fp = df[df["Threshold"].round(3) == 0.7].sum()["FP"] / predicted_objects_total[i] * 100
    false_positives.append(fp)
    i+=1

d={'FP': false_positives}
index = model_names

df_fp = pd.DataFrame(data=d, index=index)
fig_fp = df_fp.plot.bar(legend=None)
plt.suptitle('Oversegmentation')
fig_fp.set_ylabel('Percentage of false positives')
fig_fp.yaxis.set_major_formatter(mtick.PercentFormatter())
plt.xticks(rotation = 360)
plt.show()

df_fp